# Symbolic full-order model — two-mass oscillator

MORFE's full-order models are usually assembled by a finite-element backend. This example
takes the other route: it writes the equations of motion as **symbolic expressions** and lets
`model_from_symbolics` extract the linear matrices and the multilinear terms from them.

The model is the two-degree-of-freedom nonlinear oscillator of Shaw and Pierre — two coupled
masses with linear stiffness and damping, the first carrying an additional cubic restoring
force. The same machinery then builds a gallery of `ExternalSystem` drivers (harmonic,
quasi-periodic, multiharmonic and chaotic), and the last sections drive a small oscillator
with each of them.

The example uses only the public API: `model_from_symbolics` and
`externalsystem_from_symbolics`. There is no example-specific extraction, solver or printer.

Before the first run, run `julia setup.jl` in the bash from this repository. For more information look at the README.md of the repository.

`MORFESymbolicsExt` is a package extension: it loads by itself as soon as both `MORFE` and
`Symbolics` are in the same session, so there is nothing to import by name.

In [ ]:
import Pkg
Pkg.activate(@__DIR__; io = devnull)
Pkg.instantiate(; io = devnull)

using MORFE, Symbolics
using DifferentialEquations, Plots
using LinearAlgebra, Dates
using MORFE.Polynomials: evaluate

const FIGDIR = joinpath(@__DIR__, "results", "figures")
mkpath(FIGDIR)

## 1. The oscillator, without forcing

$$\ddot u_1 + c\,\dot u_1 - c\,\dot u_2 + (1+k)\,u_1 - k\,u_2 + g\,u_1^3 = 0$$
$$\ddot u_2 - c\,\dot u_1 + 2c\,\dot u_2 - k\,u_1 + (1+k)\,u_2 = 0$$

Declare one `Symbolics` vector per derivative order, write each equation as an expression that
equals zero, and hand both to `model_from_symbolics`. The `groups` tuple is ordered
**low to high**: its last entry is the derivative the equations are solved for.

In [ ]:
@variables u[1:2] du[1:2] ddu[1:2]
u, du, ddu = collect(u), collect(du), collect(ddu)

k, g, c = 1.0, 6.0, 0.1

exprs = [
    ddu[1] + c*du[1] - c*du[2] + (1 + k)*u[1] - k*u[2] + g*u[1]^3,
    ddu[2] - c*du[1] + 2*c*du[2] - k*u[1] + (1 + k)*u[2]
]

groups = (u, du, ddu)   # ascending order: (u, u̇, ü)

model = model_from_symbolics(exprs, groups)

## 2. What the extraction produced

`model_from_symbolics` differentiates `exprs` with respect to each group and evaluates the
result at the origin, giving the linear matrices $\mathbf B_0, \mathbf B_1, \mathbf B_2$.
Whatever is left after subtracting the linear part is split into monomials, grouped by
multidegree, and turned into one `MultilinearMap` per group — here the single cubic $g\,u_1^3$.

A `multiindex` is a calling convention: entry $j$ counts how many arguments carry the
$(j-1)$-th derivative. So `(3, 0)` means three copies of $\mathbf u$ and none of
$\dot{\mathbf u}$.

In [ ]:
for (j, B) in enumerate(model.linear_terms)
    println("B", j - 1, " = ", B)
end
println()
for term in model.nonlinear_terms
    println("multiindex ", term.multiindex,
        "   degree ", sum(term.multiindex),
        "   multiplicity_external ", term.multiplicity_external)
end

## 3. Harmonic forcing, as an external system

MORFE's models are autonomous, so a forcing term $-2\cos\Omega t$ is not written directly.
It is replaced by $-r_1-r_2$, where $\mathbf r$ carries its own autonomous dynamics
$\dot r_1 = i\Omega r_1$, $\dot r_2 = -i\Omega r_2$. Those have solutions
$r_j(t) = r_{j,0}e^{\pm i\Omega t}$, so $r_1+r_2 = 2\cos\Omega t$ when both start at 1.

Passing the external variables and their right-hand side as two extra arguments builds the
`ExternalSystem` internally and couples it to the model in one call. Note the argument order:
the **variables** come third, their **equations** fourth.

In [ ]:
@variables uf[1:2] duf[1:2] dduf[1:2] r[1:2]
uf, duf, dduf, r = collect(uf), collect(duf), collect(dduf), collect(r)

Ω = 1.3

exprs_forced = [
    dduf[1] + c*duf[1] - c*duf[2] + (1 + k)*uf[1] - k*uf[2] + g*uf[1]^3 - r[1] - r[2],
    dduf[2] - c*duf[1] + 2*c*duf[2] - k*uf[1] + (1 + k)*uf[2]
]

ext_exprs = [im*Ω*r[1], -im*Ω*r[2]]

model_forced = model_from_symbolics(exprs_forced, (uf, duf, dduf), r, ext_exprs)

## 4. A gallery of external systems

$\mathbf E(\mathbf r)$ must be **polynomial** and **autonomous** — $t$ may never appear, which
is the whole point of routing time dependence through an external system — and the origin must
be an equilibrium, $\mathbf E(\mathbf 0) = \mathbf 0$.

### Harmonic and quasi-periodic

The driver above, on its own. A second incommensurate pair closes onto a torus rather than a
circle: still linear, still diagonal, just twice as many states.

In [ ]:
@variables rh[1:2]
rh = collect(rh)
harmonic = externalsystem_from_symbolics([im*Ω*rh[1], -im*Ω*rh[2]], rh)

@variables rq[1:4]
rq = collect(rq)
Ω1, Ω2 = 1.0, sqrt(2)   # incommensurate

quasi = externalsystem_from_symbolics(
    [im*Ω1*rq[1], -im*Ω1*rq[2], im*Ω2*rq[3], -im*Ω2*rq[4]], rq)

### Multiharmonic — and the change of basis

Diagonal systems are not the only option. Suppose the two signals wanted are
$r_1 = -\cos\Omega t$ and $r_2 = 0.03\sin\Omega t - \cos 4\Omega t$, which are not diagonal in
any obvious basis. Introducing $r_3 = \sin\Omega t$ and $r_4 = \sin 4\Omega t$ closes them into
a first-order linear system.

Its matrix is not upper triangular, and MORFE requires that — the cohomological equations are
solved monomial by monomial in graded-lexicographic order, and a sub-diagonal entry would be
read as zero. Rather than reject the system, the constructor **re-bases** it: it finds a basis
`Q` in which the linear part is triangular, re-expresses the whole polynomial there, and
reports the change of coordinates. Watch for the `@info` message below.

In [ ]:
@variables rm[1:4]
rm = collect(rm)

multiharmonic = externalsystem_from_symbolics(
    [Ω*rm[3],
     -0.03*Ω*rm[1] + 4*Ω*rm[4],
     -Ω*rm[1],
     -4*Ω*rm[2] + 0.12*Ω*rm[3]], rm)

external_basis(multiharmonic)   # the Q relating physical r to the stored coordinates r′

### Chaotic — the Lorenz system

A nonlinear driver is fine, under the same two conditions. Lorenz has three equilibria and none
of them is the origin, so the coordinates are shifted to the non-trivial equilibrium
$C_+ = (C, C, \rho-1)$ with $C = \sqrt{\beta(\rho-1)}$ before the system is defined:

$$\dot X = \sigma(Y-X), \qquad \dot Y = X - Y - Z(X+C), \qquad \dot Z = C(X+Y)+XY-\beta Z$$

That shift is something a change of basis cannot do for you — it removes the constant term,
and an `ExternalSystem` polynomial has none.

Note that this expression method runs **no** validation: it does not check that the right-hand
side is polynomial, and it does not check that the origin is an equilibrium. A stray constant
term is absorbed silently. Those checks (`check_expr`, and through it `is_polynomial` and
`check_constant_terms`) run only on the `model_from_symbolics` path.

In [ ]:
σ, ρ, β = 10.0, 28.0, 8/3
C = sqrt(β*(ρ - 1))

@variables X Y Z

lorenz = externalsystem_from_symbolics(
    [σ*(Y - X),
     X - Y - Z*(X + C),
     C*(X + Y) + X*Y - β*Z], [X, Y, Z])

## 5. The DifferentialEquations.jl-shaped layer

Both constructors also accept a function written the way DifferentialEquations.jl expects, with
no `Symbolics` variables to declare by hand.

For `model_from_symbolics` the function must be **in-place**, and its arguments run from the
highest derivative down to $\mathbf u$: `f!(dᴺu, …, du, u, p, t)`. `p` is passed straight
through, so parameters can be supplied rather than closed over.

`externalsystem_from_symbolics` accepts either layout — in-place `g!(dr, r, p, t)` or
out-of-place `g(r, p, t) -> dr`. One restriction is worth knowing: this route builds a real
`Vector{Num}`, so the right-hand side must be **real**. The harmonic driver is written below in
its equivalent real form $\dot r_1 = \Omega r_2$, $\dot r_2 = -\Omega r_1$, which traces the
same circle.

Each object below is checked against the symbolic one built earlier.

In [ ]:
function two_mass!(a, v, x, p, t)          # a = ü, v = u̇, x = u
    kk, gg, cc = p
    a[1] = -cc*v[1] + cc*v[2] - (1 + kk)*x[1] + kk*x[2] - gg*x[1]^3
    a[2] = cc*v[1] - 2*cc*v[2] + kk*x[1] - (1 + kk)*x[2]
end

model_fn = model_from_symbolics(two_mass!, 2, 2; p = (k, g, c))

@assert all(model_fn.linear_terms[j] ≈ model.linear_terms[j] for j in 1:3)
println("function form reproduces the symbolic model's B₀, B₁, B₂")

rotation!(dr, rr, p, t) = (dr[1] = p[1]*rr[2]; dr[2] = -p[1]*rr[1])
rotation = externalsystem_from_symbolics(rotation!, 2; p = (Ω,))

@assert externalsystem_from_symbolics([Ω*rh[2], -Ω*rh[1]], rh
        ).first_order_dynamics.coefficients ≈
        rotation.first_order_dynamics.coefficients
println("p reached the driver: eigenvalues ",
    round.(diag(rotation.first_order_dynamics.coefficients[:, 1:2]), sigdigits = 4))

The coupled form takes both functions at once. `f!` then carries the external state as one
extra argument, after `u` and before `p` — referencing `r` there is how forcing enters:

```julia
f!(dᴺu, …, du, u, r, p, t)
g!(dr, r, p, t)
```

In [ ]:
function forced_two_mass!(a, v, x, rr, p, t)
    kk, gg, cc = p
    a[1] = -cc*v[1] + cc*v[2] - (1 + kk)*x[1] + kk*x[2] - gg*x[1]^3 + rr[1] + rr[2]
    a[2] = cc*v[1] - 2*cc*v[2] + kk*x[1] - (1 + kk)*x[2]
end

model_coupled = model_from_symbolics(
    forced_two_mass!, 2, 2, rotation!, 2; p = (k, g, c), p_ext = (Ω,))

println("terms reading the external state: ",
    count(t -> t.multiplicity_external > 0, model_coupled.nonlinear_terms))

## 6. Driving an oscillator with each system

Everything so far only *builds* objects. To see a driver do what it claims, integrate a plain
damped oscillator

$$\ddot y + c\,\dot y + k\,y = \mathrm{forcing}(\mathbf r), \qquad \dot{\mathbf r} = \mathbf E(\mathbf r)$$

alongside the driver's own dynamics, as one state vector `[y, ẏ, r…]`.

The driver's right-hand side is not retyped: it comes from the `ExternalSystem` itself through
`evaluate`. Two details matter for a re-based system. The initial condition is given
physically and mapped into the stored coordinates with `external_basis`, and the forcing reads
the state back with `to_physical_external` — which is exactly what MORFE does internally when
it feeds nonlinear terms the physical external argument.

In [ ]:
cc, kk = 0.2, 1.0   # a plain, lightly damped oscillator

reduced_initial(sys, r0) = (Q = external_basis(sys);
                            Q === nothing ? ComplexF64.(r0) : ComplexF64.(collect(Q \ r0)))

function drive(sys, r0_physical, forcing, tspan)
    function rhs!(ds, s, p, t)
        r = @view s[3:end]
        ds[3:end] .= evaluate(sys.first_order_dynamics, r)     # the driver, from the object
        ds[1] = s[2]
        ds[2] = -cc*s[2] - kk*s[1] + forcing(to_physical_external(sys, r))
    end
    s0 = vcat(ComplexF64(0), ComplexF64(0), reduced_initial(sys, r0_physical))
    solve(ODEProblem(rhs!, s0, tspan), Tsit5(); abstol = 1e-9, reltol = 1e-9)
end

tspan = (0.0, 40.0)

runs = [
    ("harmonic", harmonic, ComplexF64[1, 1], p -> real(p[1] + p[2])),
    ("multiharmonic", multiharmonic, ComplexF64[-1, -1, 0, 0], p -> real(p[2])),
    ("Lorenz", lorenz, ComplexF64[1, 1, 1], p -> 0.05*real(p[1]))
]

sols = [(name, drive(sys, r0, f, tspan), sys, f) for (name, sys, r0, f) in runs]
for (name, sol, _, _) in sols
    println(rpad(name, 15), sol.retcode, "  ", length(sol.t), " steps")
end

The harmonic and multiharmonic drivers have closed forms, so the integrated forcing can be
checked rather than trusted: $r_1+r_2 = 2\cos\Omega t$ and
$r_2 = 0.03\sin\Omega t - \cos 4\Omega t$.

Lorenz has no closed form, and it carries a subtlety instead. MORFE stores external dynamics in
complex coordinates, and a driver whose *physical* state is real then has a reality condition to
respect: the coordinate of a real eigenvalue stays real, and a conjugate pair stays conjugate.
Nothing in a general-purpose integrator enforces that, so the imaginary part drifts.

The condition applies to the multiharmonic and Lorenz drivers, whose physical states are real
signals. It does **not** apply to the harmonic one: there the physical $\mathbf r$ is a conjugate
pair by construction, $r_1 = e^{i\Omega t}$ and $r_2 = e^{-i\Omega t}$, so a non-zero imaginary
part is the signal rather than an error — only the sum $r_1+r_2$ is real.

In [ ]:
physical(sys, s) = to_physical_external(sys, @view s[3:end])

for (name, sol, sys, _) in sols[2:end]     # the harmonic driver is conjugate by design
    drift = maximum(maximum(abs.(imag.(physical(sys, s)))) for s in sol.u)
    println(rpad(name, 15), "reality residual max|imag r| = ", round(drift, sigdigits = 3))
end

sol_h = sols[1][2]
err_h = maximum(abs(real(sum(physical(harmonic, s))) - 2cos(Ω*t))
                for (s, t) in zip(sol_h.u, sol_h.t))
println("\nharmonic      max |r₁+r₂ − 2cos(Ωt)|           = ", err_h)

sol_m = sols[2][2]
err_m = maximum(abs(real(physical(multiharmonic, s)[2]) - (0.03sin(Ω*t) - cos(4Ω*t)))
                for (s, t) in zip(sol_m.u, sol_m.t))
println("multiharmonic max |r₂ − (0.03sinΩt − cos4Ωt)| = ", err_m)

Each panel shows the oscillator's response $y(t)$ against the raw forcing signal that produced
it. The response locks onto the shape of its driver: a clean cosine, the two-frequency wave,
and a chaotic trace.

In [ ]:
panels = map(sols) do (name, sol, sys, forcing)
    t = range(tspan[1], tspan[2]; length = 2000)
    y = [real(sol(τ)[1]) for τ in t]
    F = [forcing(to_physical_external(sys, @view sol(τ)[3:end])) for τ in t]
    plt = plot(t, y; label = "y(t)", xlabel = "t", title = name,
        lw = 1.8, color = :purple, legend = :topright)
    plot!(plt, t, F; label = "forcing(t)", ls = :dash, lw = 1.2, color = :seagreen)
    plt
end

fig = plot(panels...; layout = (3, 1), size = (900, 750), left_margin = 5Plots.mm)
savefig(fig, joinpath(FIGDIR, "forced_response.png"))
fig

## 7. Record what was built

A short run summary next to the figure, so a stored result says which model and which driver
produced it.

In [ ]:
commit = try
    readchomp(`git -C $(@__DIR__) rev-parse --short HEAD`)
catch
    "unknown"
end

open(joinpath(@__DIR__, "results", "summary.txt"), "w") do io
    println(io, "Symbolic full-order model — two-mass oscillator")
    println(io, "generated ", Dates.now())
    println(io, "julia ", VERSION, "   MORFE commit ", commit)
    println(io)
    println(io, "model: 2 dof, ORD = 2, parameters k=$k g=$g c=$c")
    for (j, B) in enumerate(model.linear_terms)
        println(io, "  B", j - 1, " = ", B)
    end
    for term in model.nonlinear_terms
        println(io, "  nonlinear term multiindex ", term.multiindex,
            ", multiplicity_external ", term.multiplicity_external)
    end
    println(io)
    println(io, "external systems")
    for (name, sys) in ("harmonic" => harmonic, "quasi-periodic" => quasi,
        "multiharmonic" => multiharmonic, "Lorenz" => lorenz)
        rebased = external_basis(sys) === nothing ? "as given" : "re-based"
        println(io, "  ", rpad(name, 16), lpad(size(
                sys.first_order_dynamics.coefficients, 1), 2), " states, ", rebased)
    end
    println(io)
    println(io, "integration checks over t ∈ ", tspan)
    println(io, "  harmonic      max |r1+r2 - 2cos(wt)|          = ", err_h)
    println(io, "  multiharmonic max |r2 - (0.03 sin - cos 4w)|  = ", err_m)
end

print(read(joinpath(@__DIR__, "results", "summary.txt"), String))